# Core Math Review

Companion notebook to [Core Math Review](https://yattas.com/tutorials/core-math-review/), a standalone companion to the [Understanding Transformers](https://yattas.com/tutorials/transformers/) series. The lesson explains scalars, vectors, dot products, and matrices; this notebook runs the same examples so you can change the numbers and see what happens.

Runs entirely on Colab's free CPU tier -- every example here finishes in milliseconds.

Use Colab's outline (View -> Table of contents) to jump between sections.


## 1. Scalars & Vectors

A scalar is one number. A vector is an ordered list of numbers -- a way to describe something with several features at once. A token embedding is a vector too, except its coordinates are learned rather than named by a person.


In [ ]:
import numpy as np

# A scalar is one number.
temperature = 72.5

# A vector is an ordered list of numbers -- several features described at once.
house = np.array([1800, 3, 12])   # [square feet, bedrooms, age]
print("house vector:", house)
print("dimension (entry count):", house.shape[0])

# A token embedding is also a vector -- same shape, different meaning. Its
# coordinates are learned rather than named by a person, so no single entry
# has a clean label the way "bedrooms" does above.
embedding = np.array([0.12, -0.87, 0.33, 0.05])
print("\nembedding vector:", embedding)
print("dimension:", embedding.shape[0])


## 2. Dot Products: A Similarity Meter

A dot product answers "how much do these two things agree?" -- multiply corresponding entries and add the results. Large and positive means the vectors point the same way, near zero means they're unrelated, negative means they point in opposite directions. This is the exact computation self-attention runs to score how relevant one token is to another.


In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

agree     = np.array([1, 1])
same_dir  = np.array([2, 2])     # points the same way as `agree`
unrelated = np.array([1, -1])    # perpendicular to `agree`
opposite  = np.array([-1, -1])   # points the opposite way

for name, v in [("same direction", same_dir), ("unrelated", unrelated), ("opposite", opposite)]:
    print(f"{name:>15}: dot(agree, v) = {dot(agree, v):>3}   (np.dot: {np.dot(agree, v)})")


## 3. Matrices: Machines That Rewrite Descriptions

If a vector is a description, a matrix is a machine that rewrites descriptions -- it takes a vector in and produces a different vector out. Every layer in a neural network is one or more of these. Multiplying `W` (m rows, n columns) by a vector `x` (n entries) produces a new vector `y` (m entries): each entry of `y` is one row of `W` dotted with `x`.


In [ ]:
W = np.array([[1, 2],
              [0, 1]])
x = np.array([3, 2])

y = W @ x
print("W @ x =", y)   # [1*3 + 2*2, 0*3 + 1*2] = [7, 2]


**Dimensions are a built-in error check.** If `W` is `m x n`, it only accepts an `n`-entry vector as input. Feeding it the wrong shape fails immediately instead of silently computing a wrong answer:


In [ ]:
bad_x = np.array([3, 2, 1])
try:
    W @ bad_x
except ValueError as e:
    print("Shape mismatch, as expected:", e)


## 4. Where This Shows Up: A Toy Attention Score

Self-attention scores every token pair with a dot product between a Query vector and a Key vector. A higher score means "pay more attention to this token." Below, one query is closer in direction to one of three keys -- the dot product picks it out.


In [ ]:
query = np.array([1.0, 0.0, 1.0])

keys = {
    "cat":  np.array([0.9, 0.1, 0.8]),    # close to the query's direction
    "moon": np.array([0.1, 0.9, -0.2]),   # far from it
    "roof": np.array([-0.5, 0.2, -0.9]),  # points mostly the opposite way
}

scores = {name: np.dot(query, k) for name, k in keys.items()}
for name, score in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"{name:>5}: {score:.2f}")
